#### 1. INSTALAÇÃO DE DEPENDÊNCIAS

In [1]:
!pip install -r ../requirements.txt

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   --------------------------- ------------ 6.8/9.9 MB 34.7 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 33.4 MB/s  0:00:00

   -------- ------------------------------- 1/5 [plotly]
   -------- ------------------------------- 1/5 [plotly]
   -------- ------------------------------- 1/5 [plotly]
   -------- ------------------------------- 1/5 [plotly]
   -------- ------------------------------- 1/5 [plotly]
   -------- ------------------------------- 1/5 [plotly]
   -------- ------------------------------- 1/5 [plotly]
   -------- ------------------------------- 1/5 [plotly]
   -------- ------------------------------- 1/5 [plotly]
   -------- ------------------------------- 1/5 [plotly]
   -------- ------------------------------- 1/5 [plotly]
   -------- ------------------------------- 1/5 [plotly]
   -------- ------------------------------- 1/5 [plotly]
   -------- --------------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


#### 2. IMPORTAÇÕES e CONFIGURAÇÕES 

In [4]:
import glob
from pathlib import Path

import pandas as pd
from tqdm import tqdm

# Diretórios
RAW_PATH = Path("../data/raw")
PROCESSED_PATH = Path("../data/processed")

# Arquivo de saída
OUTPUT_FILE = PROCESSED_PATH / "exportacoes_tratadas.csv"

#### 3. FUNÇÕES AUXILIARES 

In [5]:
def carregar_arquivos_csv(caminho):
    """
    Lê todos os arquivos CSV da pasta e concatena em um único DataFrame.
    """
    
    arquivos = glob.glob(str(caminho / "*.csv"))

    print(f"Arquivos encontrados: {len(arquivos)}")

    dfs = []

    for arquivo in tqdm(arquivos):
        df_temp = pd.read_csv(
            arquivo,
            sep=";",
            low_memory=False
        )

        # Extrai o ano do nome do arquivo
        ano = Path(arquivo).stem.split("_")[-1]

        df_temp["ano_arquivo"] = ano

        dfs.append(df_temp)

    return pd.concat(dfs, ignore_index=True)


def padronizar_colunas(df):
    """
    Padroniza os nomes das colunas.
    """

    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
    )

    return df


def remover_duplicados(df):
    """
    Remove registros duplicados.
    """

    qtd = df.duplicated().sum()

    print(f"Duplicados encontrados: {qtd}")

    return df.drop_duplicates()


def resumo_dataset(df):
    """
    Exibe informações gerais do dataset.
    """

    print("\nShape:", df.shape)

    print("\nTipos de dados:")
    display(df.dtypes)

    print("\nValores nulos:")
    display(
        df.isnull()
          .sum()
          .sort_values(ascending=False)
          .head(20)
    )

#### 4. EXTRAÇÃO E LIMPEZA 

In [6]:
df = carregar_arquivos_csv(RAW_PATH)

df = padronizar_colunas(df)

df = remover_duplicados(df)

resumo_dataset(df)

df.head()

Arquivos encontrados: 4


100%|██████████| 4/4 [00:09<00:00,  2.31s/it]


Duplicados encontrados: 0

Shape: (5561220, 12)

Tipos de dados:


co_ano         int64
co_mes         int64
co_ncm         int64
co_unid        int64
co_pais        int64
sg_uf_ncm        str
co_via         int64
co_urf         int64
qt_estat       int64
kg_liquido     int64
vl_fob         int64
ano_arquivo      str
dtype: object


Valores nulos:


co_ano         0
co_mes         0
co_ncm         0
co_unid        0
co_pais        0
sg_uf_ncm      0
co_via         0
co_urf         0
qt_estat       0
kg_liquido     0
vl_fob         0
ano_arquivo    0
dtype: int64

,co_ano,co_mes,co_ncm,co_unid,co_pais,sg_uf_ncm,co_via,co_urf,qt_estat,kg_liquido,vl_fob,ano_arquivo
0,2023,3,84139190,10,160,RJ,4,817700,107365,5,20,2023
1,2023,12,73182100,10,764,SP,1,817800,0,0,84,2023
2,2023,4,84219999,10,40,RS,4,817600,2,2,758,2023
3,2023,5,85291090,10,63,RJ,7,1017503,1,1,275,2023
4,2023,10,84825010,11,63,MG,7,1017503,2,0,9,2023


#### 5. TRANSFORMAÇÕES 

In [7]:
# Exemplo de conversão numérica
colunas_numericas = [
    "vl_fob",
    "kg_liquido"
]

for coluna in colunas_numericas:

    if coluna in df.columns:

        df[coluna] = (
            pd.to_numeric(
                df[coluna],
                errors="coerce"
            )
        )

# Valor FOB por KG

if {"vl_fob", "kg_liquido"}.issubset(df.columns):

    df["valor_medio_kg"] = (
        df["vl_fob"] /
        df["kg_liquido"]
    )

# Tratar divisões por zero

    df["valor_medio_kg"] = (
        df["valor_medio_kg"]
        .replace([float("inf"), -float("inf")], pd.NA)
    )

#### 6. QUALIDADE DOS DADOS

In [8]:
print("Quantidade de registros:", len(df))

print("Quantidade de colunas:", len(df.columns))

print("\nPercentual de nulos:")

display(
    (
        df.isnull()
          .mean()
          .mul(100)
          .round(2)
          .sort_values(ascending=False)
    )
)

Quantidade de registros: 5561220
Quantidade de colunas: 13

Percentual de nulos:


valor_medio_kg    7.69
co_mes            0.00
co_ncm            0.00
co_unid           0.00
co_ano            0.00
co_pais           0.00
sg_uf_ncm         0.00
co_urf            0.00
co_via            0.00
qt_estat          0.00
kg_liquido        0.00
vl_fob            0.00
ano_arquivo       0.00
dtype: float64

#### 7. SALVAR CAMADA PROCESSED

In [10]:
PROCESSED_PATH.mkdir(
    parents=True,
    exist_ok=True
)

df.to_parquet(
    "../data/processed/exportacoes.parquet",
    index=False
)

print(f"Arquivo salvo em: {OUTPUT_FILE}")

print("Processo concluído com sucesso.")

Arquivo salvo em: ..\data\processed\exportacoes_tratadas.csv
Processo concluído com sucesso.
